# Lesson 4: Chaining Prompts for Agentic Reasoning

## Automated Claim Triage: From First-Notice to the Right Queue

In this hands-on exercise, you will build a prompt chain that extracts key fields from free-form auto-claim reports, assesses damage severity, and routes each claim to one of several queues — `glass`, `fast_track`, `material_damage`, or `total_loss` — with code-based gate checks at every step.

## Outline:

- Setup
- Sample FNOL (First Notice of Loss) Texts
- Stage I: Information Extraction
- Stage II: Severity Assessment
- Stage III: Queue Routing
- Review Outputs

## Setup

Import necessary libraries and define helper functions, including a mock LLM client, code execution environment, and test runner.

In [1]:
# Import necessary libraries
# No changes needed in this cell
import anthropic  # For accessing the Claude API
from dotenv import load_dotenv, find_dotenv  # For loading API keys from .env
import json
from pydantic import BaseModel, Field  # For structured data validation
from typing import List, Literal, Optional

In [2]:
# Set up LLM credentials
# Reads ANTHROPIC_API_KEY and ANTHROPIC_WORKSPACE_ID from the .env file in the Udacity folder

import os

load_dotenv(find_dotenv(usecwd=True), override=True)

# User-level keys (sk-ant-usr-...) must name their workspace on every request
client = anthropic.Anthropic(
    default_headers={"anthropic-workspace-id": os.environ["ANTHROPIC_WORKSPACE_ID"]}
)

In [3]:
# Define helper functions
# No changes needed in this cell

MODEL = "claude-haiku-5-5"  # Small, cheap model; plenty for these short JSON tasks


def get_completion(messages=None, system_prompt=None, user_prompt=None, model=MODEL):
    """
    Function to get a completion from the Claude API.
    Args:
        messages: Message list (a "system" message is passed as Claude's system prompt)
        system_prompt: The system prompt
        user_prompt: The user prompt
        model: The model to use (default is claude-haiku-5-5)
    Returns:
        The completion text (JSON string)
    """

    messages = list(messages or [])
    # Claude takes the system prompt as a separate argument, not as a message
    system = system_prompt or ""
    chat = []
    for m in messages:
        if m["role"] == "system":
            system = m["content"]
        else:
            chat.append(m)
    if user_prompt:
        chat.append({"role": "user", "content": user_prompt})

    response = client.messages.create(
        model=model,
        max_tokens=16000,
        system=system,
        messages=chat,
    )
    text = "".join(b.text for b in response.content if b.type == "text").strip()
    # Claude has no JSON mode, so strip ```json fences before the gates call json.loads()
    if text.startswith("```"):
        text = text.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return text

## Sample FNOL (First Notice of Loss) Texts
Let's define a few sample First Notice of Loss (FNOL) texts to process through our chain.

In [4]:
# Define sample FNOL texts
# TODO: [Optional] Add more sample FNOL texts to test various scenarios

sample_fnols = [
    """
    Claim ID: C001
    Customer: Priya Patel
    Vehicle: 2017 Mazda CX-5
    Incident: I was following a construction truck when a piece of gravel flew off and struck my
    windshield, leaving a small star-shaped chip on the passenger side. Nothing else was damaged.
    """,
    """
    Claim ID: C002
    Customer: Lucas Moreau
    Vehicle: 2021 Hyundai Elantra
    Incident: In stop-and-go traffic I rear-ended the car in front of me at low speed. My front
    bumper is cracked and pushed in, and the grille is broken. The car still drives normally.
    """,
    """
    Claim ID: C003
    Customer: Aisha Okafor
    Vehicle: 2023 Chevrolet Silverado
    Incident: I hit a deer at highway speed at night. The entire front end is crushed, the hood is
    folded, the radiator is leaking coolant, and the engine will not start. The truck had to be towed.
    """,
    """
    Claim ID: C004
    Customer: Tomasz Kowalski
    Vehicle: 2016 Volkswagen Golf
    Incident: During a storm a large tree branch fell onto my parked car. The roof and trunk lid are
    dented, the rear quarter panel is bent, and the rear window has a long crack.
    """,
    """
    Claim ID: C005
    Customer: Sofia Lindqvist
    Vehicle: 2022 Kia Sportage
    Incident: A runaway shopping cart rolled into my car in a supermarket parking lot. It left a
    shallow dent and some paint scratches on the front passenger door.
    """,
]

## Stage I: Information Extraction
In this stage, we'll create a prompt that extracts structured information from free-form FNOL text.

In [6]:
# Define a system prompt for information extraction according to the provided ClaimInformation class
# TODO: Complete the prompt by replacing the parts marked with **********


class ClaimInformation(BaseModel):
    claim_id: str = Field(..., min_length=2, max_length=10)
    name: str = Field(..., min_length=2, max_length=100)
    vehicle: str = Field(..., min_length=2, max_length=100)
    loss_desc: str = Field(..., min_length=10, max_length=500)
    damage_area: List[
        Literal[
            "windshield",
            "front",
            "rear",
            "side",
            "roof",
            "hood",
            "door",
            "bumper",
            "fender",
            "quarter panel",
            "trunk",
            "glass",
        ]
    ] = Field(..., min_items=1)


info_extraction_system_prompt = """
You are an auto insurance claim processing assistant. Your task is to extract key information from First Notice of Loss (FNOL) reports.

Format your response as a valid JSON object with the following keys:
- claim_id (str): The claim ID
- name (str): The customer's full name
- vehicle (str): The vehicle make, model, and year
- loss_desc (str): A concise description of the incident
- damage_area (list[str]): A list of damaged areas on the vehicle (at least one of:
    - windshield
    - front
    - rear
    - side
    - roof
    - hood
    - door
    - bumper
    - fender
    - quarter panel
    - trunk
    - glass

For damage_area, only use items from the list above.

Only respond with the JSON object, nothing else.
"""

In [7]:
# Define a gate check function and claim extraction function
# TODO: Complete the prompt by replacing the parts marked with **********


def gate1_validate_claim_info(claim_info_json: str) -> ClaimInformation:
    """
    Gate 1: Validates claim information extracted from FNOL text.
    Returns validated ClaimInformation object or raises validation error.
    """
    try:
        # Parse the JSON string
        claim_info_dict = json.loads(claim_info_json)
        # Validate with Pydantic model
        validated_info = ClaimInformation(**claim_info_dict)
        return validated_info
    except Exception as e:
        raise ValueError(f"Gate 1 validation failed: {str(e)}")


def extract_claim_info(fnol_text):
    """
    Stage 1: Extract structured information from FNOL text
    """
    messages = [
        {"role": "system", "content": info_extraction_system_prompt},
        {"role": "user", "content": fnol_text},
    ]

    response = get_completion(messages=messages)

    # Gate check: validate the extracted information
    try:
        validated_info = gate1_validate_claim_info(response)
        return validated_info
    except ValueError as e:
        print(f"Gate 1 failed: {e}")
        return None

In [8]:
# Run the claim extraction function on the sample FNOLs
# No updates needed in this cell

extracted_claim_info_items = [
    extract_claim_info(fnol_text) for fnol_text in sample_fnols
]
extracted_claim_info_items

[ClaimInformation(claim_id='C001', name='Priya Patel', vehicle='2017 Mazda CX-5', loss_desc='Gravel thrown from a construction truck struck the windshield, causing a small star-shaped chip on the passenger side. No other damage reported.', damage_area=['windshield']),
 ClaimInformation(claim_id='C002', name='Lucas Moreau', vehicle='2021 Hyundai Elantra', loss_desc='Low-speed rear-end collision in stop-and-go traffic; insured struck the vehicle ahead, cracking and pushing in the front bumper and breaking the grille. Vehicle remains drivable.', damage_area=['front', 'bumper']),
 ClaimInformation(claim_id='C003', name='Aisha Okafor', vehicle='2023 Chevrolet Silverado', loss_desc='Collision with a deer at highway speed at night. Front end crushed, radiator leaking coolant, engine will not start, vehicle towed.', damage_area=['front', 'hood']),
 ClaimInformation(claim_id='C004', name='Tomasz Kowalski', vehicle='2016 Volkswagen Golf', loss_desc="A large tree branch fell onto the insured's pa

## Stage II: Severity Assessment
In this stage, we'll assess the severity of the damage based on the extracted information.

Note, our carrier applies the following heuristics:
- Minor damage: Small dents, scratches, glass chips (cost range: $100-$1,000)
- Moderate damage: Single panel damage, bumper replacement, door damage (cost range: $1,000-$5,000)
- Major damage: Structural damage, multiple panel replacement, engine/drivetrain issues, total loss candidates (cost range: $5,000-$50,000)

In this example we will let the LLM estimate the cost, though in production we would want a more accurate estimate, e.g. querying a database of repair costs.


In [9]:
# Define a system prompt for severity assessment according to the provided SeverityAssessment class
# TODO: Complete the prompt by replacing the parts marked with **********


class SeverityAssessment(BaseModel):
    severity: Literal["Minor", "Moderate", "Major"]
    est_cost: float = Field(..., gt=0)


severity_assessment_system_prompt = """
You are an auto insurance damage assessor. Your task is to evaluate the severity of vehicle damage and estimate repair costs.

Apply these carrier heuristics:
- Minor damage: Small dents, scratches, glass chips (cost range: $100-$1,000)
- Moderate damage: Single panel damage, bumper replacement, door damage (cost range: $1,000-$5,000)
- Major damage: Structural damage, multiple panel replacement, engine/drivetrain issues, total loss candidates (cost range: $5,000-$50,000)

Based on the claim information provided, determine:
1. Severity level (Minor, Moderate, or Major)
2. Estimated repair cost (in USD)

Format your response as a valid JSON object with the following keys:
- severity: One of "Minor", "Moderate", or "Major"
- est_cost: Numeric estimate of repair costs (e.g., 750.00)

Only respond with the JSON object, nothing else.
"""

In [10]:
# Define a gate check function and assess_severity function
# TODO: Complete the prompt by replacing the parts marked with **********


def gate2_cost_range_ok(severity_json: str) -> SeverityAssessment:
    """
    Gate 2: Validates that the estimated cost is within reasonable range for the severity.
    Returns validated SeverityAssessment object or raises validation error.
    """
    try:
        # Parse the JSON string
        severity_dict = json.loads(severity_json)
        # Validate with Pydantic model
        validated_severity = SeverityAssessment(**severity_dict)

        # Check cost range based on severity
        if validated_severity.severity == "Minor" and (
            validated_severity.est_cost < 100 or validated_severity.est_cost > 1000
        ):
            raise ValueError(
                f"Minor damage should cost between $100-$1000, got ${validated_severity.est_cost}"
            )
        elif validated_severity.severity == "Moderate" and (
            validated_severity.est_cost < 1000 or validated_severity.est_cost > 5000
        ):
            raise ValueError(
                f"Moderate damage should cost between $1000-$5000, got ${validated_severity.est_cost}"
            )
        elif validated_severity.severity == "Major" and (
            validated_severity.est_cost < 5000 or validated_severity.est_cost > 50000
        ):
            raise ValueError(
                f"Major damage should cost between $5000-$50000, got ${validated_severity.est_cost}"
            )

        return validated_severity
    except Exception as e:
        raise ValueError(f"Gate 2 validation failed: {str(e)}")


def assess_severity(claim_info: ClaimInformation) -> Optional[SeverityAssessment]:
    """
    Stage 2: Assess severity based on damage description
    """

    # Convert Pydantic model to JSON string
    claim_info_json = claim_info.model_dump_json()

    messages = [
        {"role": "system", "content": severity_assessment_system_prompt},
        {"role": "user", "content": claim_info_json},
    ]

    response = get_completion(messages=messages)

    # Gate check: validate the severity assessment
    try:
        validated_severity = gate2_cost_range_ok(response)
        return validated_severity
    except ValueError as e:
        print(f"Gate 2 failed: {e}. Response: {response}")
        return None


In [17]:
# Run the claim extraction function on the sample data
# No updates needed in this cell

severity_assessment_items = [
    assess_severity(item) for item in extracted_claim_info_items
]

severity_assessment_items

Gate 2 failed: Gate 2 validation failed: Major damage should cost between $5000-$50000, got $4800.0. Response: {"severity": "Major", "est_cost": 4800.00}


[SeverityAssessment(severity='Minor', est_cost=350.0),
 SeverityAssessment(severity='Moderate', est_cost=2200.0),
 SeverityAssessment(severity='Major', est_cost=11250.0),
 None,
 SeverityAssessment(severity='Moderate', est_cost=1200.0)]

## 6. Stage III: Queue Routing
In this stage, we'll route the claim to the appropriate queue based on severity and damage area.

Use these routing rules:
- 'glass' queue: For Minor damage involving ONLY glass (windshield, windows)
- 'fast_track' queue: For other Minor damage
- 'material_damage' queue: For all Moderate damage
- 'total_loss' queue: For all Major damage

These are the priority levels:
- Priority 1 (highest): Safety issues, customer stranded
- Priority 2: Significant but contained damage, vehicle drivable
- Priority 3: Standard claims
- Priority 4: Minor issues, no mobility impact
- Priority 5 (lowest): Cosmetic only, no functional impact

In [12]:
# Define a system prompt for claim routing according to the provided ClaimRouting class
# TODO: Complete the prompt by replacing the parts marked with **********


class ClaimRouting(BaseModel):
    claim_id: str
    queue: Literal["glass", "fast_track", "material_damage", "total_loss"]
    priority: int = Field(..., ge=1, le=5)


queue_routing_system_prompt = """
You are an auto insurance claim routing specialist. Your task is to determine the appropriate processing queue for each claim.

Use these routing rules:
- 'glass' queue: For Minor damage involving ONLY glass (windshield, windows)
- 'fast_track' queue: For other Minor damage
- 'material_damage' queue: For all Moderate damage
- 'total_loss' queue: For all Major damage

Also assign a priority level:
- Priority 1 (highest): Safety issues, customer stranded
- Priority 2: Significant but contained damage, vehicle drivable
- Priority 3: Standard claims
- Priority 4: Minor issues, no mobility impact
- Priority 5 (lowest): Cosmetic only, no functional impact

Format your response as a valid JSON object with the following keys:
- claim_id: Use the provided claim ID
- queue: One of "glass", "fast_track", "material_damage", or "total_loss"
- priority: Integer from 1 to 5

Only respond with the JSON object, nothing else.
"""

In [13]:
# Define a gate check function and assess_severity function
# TODO: Complete the prompt by replacing the parts marked with **********


def gate3_validate_routing(routing_json: str) -> ClaimRouting:
    """
    Gate 3: Validates that the claim is routed to a valid queue.
    Returns validated ClaimRouting object or raises validation error.
    """
    try:
        # Parse the JSON string
        routing_dict = json.loads(routing_json)
        # Validate with Pydantic model
        validated_routing = ClaimRouting(**routing_dict)
        return validated_routing
    except Exception as e:
        raise ValueError(f"Gate 3 validation failed: {str(e)}")


def route_claim(
    claim_info: ClaimInformation, severity_assessment: Optional[SeverityAssessment]
) -> Optional[ClaimRouting]:
    """
    Stage 3: Route claim to appropriate queue
    """
    if severity_assessment is None:
        return None

    # Create input for the routing model
    routing_input = {
        "claim_info": claim_info.model_dump(),
        "severity_assessment": severity_assessment.model_dump(),
    }

    messages = [
        {"role": "system", "content": queue_routing_system_prompt},
        {"role": "user", "content": json.dumps(routing_input)},
    ]

    response = get_completion(messages=messages)

    # Gate check: validate the routing decision
    try:
        validated_routing = gate3_validate_routing(response)
        return validated_routing
    except ValueError as e:
        print(f"Gate 3 failed: {e}. Response: {response}")
        return None

In [14]:
# Run the routing function on the sample data
# No updates needed in this cell

routed_claim_items = [
    route_claim(claim, severity_assessment)
    for claim, severity_assessment in zip(
        extracted_claim_info_items, severity_assessment_items
    )
]

routed_claim_items

[ClaimRouting(claim_id='C001', queue='glass', priority=4),
 ClaimRouting(claim_id='C002', queue='material_damage', priority=2),
 ClaimRouting(claim_id='C003', queue='total_loss', priority=1),
 ClaimRouting(claim_id='C004', queue='total_loss', priority=3),
 ClaimRouting(claim_id='C005', queue='material_damage', priority=3)]

## 7. Review Outputs

Let's put our data into a pandas dataframe for easier analysis.

In [15]:
# No updates needed in this cell

import pandas as pd

records = []
for claim, severity_assessment, routed_claim in zip(
    extracted_claim_info_items, severity_assessment_items, routed_claim_items
):
    record = {}
    record.update(claim)
    record.update(severity_assessment)
    record.update(routed_claim)
    records.append(record)


# Show the entire dataframe since it is not too large
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
df = pd.DataFrame(records)

df

,claim_id,name,vehicle,loss_desc,damage_area,severity,est_cost,queue,priority
0,C001,Priya Patel,2017 Mazda CX-5,"Gravel thrown from a construction truck struck the windshield, causing a small star-shaped chip on the passenger side. No other damage reported.",[windshield],Minor,350.0,glass,4
1,C002,Lucas Moreau,2021 Hyundai Elantra,"Low-speed rear-end collision in stop-and-go traffic; insured struck the vehicle ahead, cracking and pushing in the front bumper and breaking the grille. Vehicle remains drivable.","[front, bumper]",Moderate,2200.0,material_damage,2
2,C003,Aisha Okafor,2023 Chevrolet Silverado,"Collision with a deer at highway speed at night. Front end crushed, radiator leaking coolant, engine will not start, vehicle towed.","[front, hood]",Major,12500.0,total_loss,1
3,C004,Tomasz Kowalski,2016 Volkswagen Golf,"A large tree branch fell onto the insured's parked vehicle during a storm, causing dents to the roof and trunk lid, a bent rear quarter panel, and a long crack in the rear window.","[roof, trunk, quarter panel, glass]",Major,5400.0,total_loss,3
4,C005,Sofia Lindqvist,2022 Kia Sportage,"A runaway shopping cart rolled into the vehicle in a supermarket parking lot, causing a shallow dent and paint scratches on the front passenger door.",[door],Moderate,1200.0,material_damage,3


## 8. Reflection & Transfer
Reflect on the effectiveness of chaining prompts for this task:

1. Prompt Chain Architecture:
* How does breaking down the task into stages affect the performance?
* What are the benefits of having gate checks between stages?
* How could the chain design be improved?

2. Error Handling:
* What types of errors might occur at each stage?
* How could we make the chain more robust against errors?
* What would a good fallback strategy look like?

3. Scalability:
* How well would this approach scale to handle more complex claims?
* What challenges might arise when processing thousands of claims?
* How could the prompt chain be optimized for efficiency?

4. Transfer to Other Domains:
* How could this prompt chaining approach be applied to other domains?
* What general principles of prompt chaining can we extract from this exercise?

## Summary

🎉 Congratulations! 🎉 You've built an impressive prompt chain system for insurance claims!
You transformed messy FNOL text into structured data, assessed damage severity, and routed claims to the right queues, all with robust gate checks! 🚀✨

Remember:

- 🔗 Chained prompts break complex tasks into manageable steps
- 🛡️ Gate checks prevent error cascades
- 🧠 Having specialized prompts helps keep code focused and maintainable

You've mastered a powerful pattern for countless business processes! 🏆
Amazing work on your agentic reasoning system! 💯